### **🪜 Step 1: Data Loading**

Before we can analyze anything, we need to get our data into a format we can work with. Here we connect to the university's student database, extract all 500 student records, and take a first look at what information we have — including what columns exist, what data types they are, and where values might be missing.

In [ ]:
import sqlite3
import os
import pandas as pd
from google.colab import files

In [ ]:
# 1. Upload the ACTUAL .db file from your computer
print("Please select your .db file:")
uploaded = files.upload()

# 2. Get the exact name of the file you just uploaded
file_name = list(uploaded.keys())[0]
print(f"Successfully uploaded: {file_name}")

# 3. Connect and Query
try:
    conn = sqlite3.connect(file_name)
    query = "SELECT * FROM student_data"
    df = pd.read_sql_query(query, conn)

    print("\n--- Success! Data Preview ---")
    # Inspect the first few rows of the dataset
    print(df.head())

    # Check for missing values and data types
    print(df.info())

    conn.close()
except Exception as e:
    print(f"\nError: {e}")
    print("Double-check that the file you uploaded is a true SQLite database file.")

Old file cleared.
Please select your .db file:


Saving student_data.db to student_data.db
Successfully uploaded: student_data.db

--- Success! Data Preview ---
   Unnamed: 0     sex   age   birthdate  country  logged in  lessons  \
0           0    None  24.0  2000-06-30  Austrai   5.297479      4.0   
1           1  Female  21.0  2002-11-29  Germany   3.051044      1.0   
2           2    Male  21.0  2003-03-25    Other  25.042989      5.0   
3           3  Female  21.0  2003-03-25   France   6.482670      1.0   
4           4  Female  29.0  1994-12-28  Germany   9.786313      1.0   

   assignments  posts mentoring      score  orientation  
0          1.0    3.0       Yes  44.452539          1.0  
1          0.0    6.0        No  18.985095          0.0  
2          1.0    8.0       Yes  52.182803          0.0  
3          0.0    3.0        No  24.172925          1.0  
4          0.0   10.0        No  39.749603          1.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #

### 🔎 🗂️ **Step 2.1 Exploratory Data Analysis**

Before building any models, we need to understand and clean our data. This section fixes inconsistent labels (like "F" vs "Female"), checks for errors and missing values, and explores each feature through statistics and visualizations — so we know exactly what we're working with before any predictions are made.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# I. Define the mapping for inconsistent country spellings
country_corrections = {
    'Austrai': 'Austria',
    'Grmany': 'Germany',
    'Othre': 'Other',
    'Itali': 'Italy',
    'Franc': 'France',
}

# Apply the corrections to the 'country' column
df['country'] = df['country'].replace(country_corrections)

# Display the unique values of the 'country' column after cleaning to verify
print(f"Unique countries after correction: {df['country'].unique()}")

# II. Define the mapping for inconsistent 'sex' values
sex_corrections = {
    'F': 'Female',
    'M': 'Male',
    'N': 'Non-Binary'
}

# Apply the corrections to the 'sex' column
df['sex'] = df['sex'].replace(sex_corrections)

# Display the unique values of the 'sex' column after cleaning to verify
print(f"Unique values in 'sex' column after correction: {df['sex'].unique()}")

# III. Define the mapping for inconsistent 'sex' values
mentoring_corrections = {
    'Y': 'Yes',
    'N': 'No',
}

# Apply the corrections to the 'mentoring' column
df['mentoring'] = df['mentoring'].replace(mentoring_corrections)

# Display the unique values of the 'mentoring' column after cleaning to verify
print(f"Unique values in 'mentoring' column after correction: {df['mentoring'].unique()}")

In [ ]:
# 1. Extracting birth_year from birthdate column for the plot
# We convert birthdate to datetime just to get the year for the X-axis
df['birth_year'] = pd.to_datetime(df['birthdate'], errors='coerce').dt.year

# 2. Scatter plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='birth_year', y='age', color='orange')
plt.title('Correlation: Birth Year vs. Age')
plt.xlabel('Year of Birth')
plt.ylabel('Age')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 3. Calculate the actual correlation number
correlation = df['age'].corr(df['birth_year'])
print(f"The correlation between Age and Birth Year is: {correlation:.2f}")

The correlation between Age and Birth Year is: -1.00


#### 📊 **Observations for Birthdate:**

By plotting `Age` vs. `Birth Year`, I identified a perfect negative correlation ($r = -1.00$), **confirming redundancy**.

Furthermore, the birthdate column contained extreme outliers (birth years near 1900) that would require complex manual correction.

To maintain data integrity and avoid making assumptions about the study's reference year, **I am dropping birthdate and keeping age as the primary feature**, which I will then clean in the following codeblocks using median imputation and IQR to handle the unusual outliers.

In [ ]:
df = df.drop(columns=['birthdate', 'birth_year'], errors='ignore')

pd.DataFrame(df.columns)

,0
0,Unnamed: 0
1,sex
2,age
3,country
4,logged in
5,lessons
6,assignments
7,posts
8,mentoring
9,score


#### 📉 **Impute Missing Values for Categorical Columns**

In [ ]:
# Impute missing values with mode for each categorical column
df['sex'] = df['sex'].fillna(df['sex'].mode()[0])
df['country'] = df['country'].fillna(df['country'].mode()[0])
df['mentoring'] = df['mentoring'].fillna(df['mentoring'].mode()[0])

# Verify the changes
print(df[['sex', 'country', 'mentoring']].isnull().sum())  # Should show 0 for these columns

sex          0
country      0
mentoring    0
dtype: int64


### 🔎 🗂️ **Step 2.2 Descriptive Statistics, Visualization, Imputation, and Handling of Outliers**

Here we take a closer look at each individual variable — age, logins, lessons, posts, score, assignments, and orientation. For each one, we calculate summary statistics, visualise its distribution, fill in any missing values, and cap extreme outliers. This ensures every feature is clean, consistent, and ready for modelling without unusual values skewing our results.

### 🧩 Age

In [ ]:
# 1. Key statistics for original 'age'
age_mean = df['age'].mean()
age_median = df['age'].median()
age_mode = df['age'].mode()[0] if not df['age'].mode().empty else None
age_std = df['age'].std()
age_min, age_max = df['age'].min(), df['age'].max()
age_range = age_max - age_min

print("=== Original Age Descriptive Statistics ===")
print(f"Mean: {age_mean:.3f}")
print(f"Median: {age_median:.3f}")
print(f"Mode: {age_mode}")
print(f"Standard Deviation: {age_std:.3f}")
print(f"Range: {age_range:.3f} (Min: {age_min}, Max: {age_max})")
print("\nFull describe:")
print(df['age'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot: clearly shows outliers as points
sns.boxplot(data=df, x='age', orient='h', ax=axes[0], color='pink')
axes[0].set_title('Original Age: Boxplot (IQR Outliers Visible)')

# Histogram: shows shape, skew, and binning
df['age'].hist(bins=30, ax=axes[1], edgecolor='black')
axes[1].axvline(age_mean, color='red', linestyle='--', label=f'Mean: {age_mean:.1f}')
axes[1].axvline(age_median, color='green', linestyle='--', label=f'Median: {age_median:.1f}')
axes[1].legend()
axes[1].set_title('Original Age: Histogram')

plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Check current missing values
print("Missing in age:", df['age'].isnull().sum())

# Step 2: Impute age NaNs with median (robust to outliers)
age_median = df['age'].median()
df['age_imputed'] = df['age'].fillna(age_median)
print(f"Imputed with median: {age_median}")

# Step 3: Now compute IQR on imputed (no NaNs)
Q1 = df['age_imputed'].quantile(0.25)
Q3 = df['age_imputed'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")

# Step 4: Cap outliers on imputed data
df['age_capped_iqr'] = df['age_imputed'].clip(lower=lower_bound, upper=upper_bound)

# Verify: no NaNs now
print("Missing in age_capped_iqr:", df['age_capped_iqr'].isnull().sum())  # Should be 0

Missing in age: 6
Imputed with median: 22.0
IQR bounds: [12.5, 32.5]
Missing in age_capped_iqr: 0


In [ ]:
# Capped age (without extreme outliers)
print('Full describe of capped age:')
print(df['age_capped_iqr'].describe())

print('=== Capped Age Descriptive Statistics ===')
print("Mean: ", df['age_capped_iqr'].mean())
print("Median: ", df['age_capped_iqr'].median())
print("Mode: ", df['age_capped_iqr'].mode())
print("Standard Deviation: ", df['age_capped_iqr'].std())
print("Range: ", df['age_capped_iqr'].max() - df['age_capped_iqr'].min())

Full describe of capped age:
count    500.000000
mean      22.703000
std        3.874832
min       16.000000
25%       20.000000
50%       22.000000
75%       25.000000
max       32.500000
Name: age_capped_iqr, dtype: float64
=== Capped Age Descriptive Statistics ===
Mean:  22.703
Median:  22.0
Mode:  0    20.0
Name: age_capped_iqr, dtype: float64
Standard Deviation:  3.874831562697736
Range:  16.5


#### 📊 **Observations for Age:**

* **Mean vs. Median:** Mean (22.703) is now very close to median (22.0), indicating minimal right skew after capping extreme high ages. This symmetry suggests effective outlier treatment, with values no longer pulling the mean upward.

* **Mode:** The most frequent age remains 20.0, typical for a student dataset clustered around early 20s. Capping preserves common values without alteration.

* **Variability:** Standard deviation (3.87) is now low and reasonable for ages, down significantly from pre-correction levels. Range (16.5) reflects bounds like ~16-32.5 post-IQR clipping, eliminating extremes.

* **Shape & Spread:** Distribution shows mild right skew, with most students likely between 18-28 after capping (bounds ~16-32.5). Variability is now resolved (low SD), enabling reliable modeling.

### 🧩 Logged In

In [ ]:
logged_mean = df['logged in'].mean()
logged_median = df['logged in'].median()
logged_mode = df['logged in'].mode()[0] if not df['logged in'].mode().empty else None
logged_std = df['logged in'].std()
logged_min, logged_max = df['logged in'].min(), df['logged in'].max()
logged_range = logged_max - logged_min

print("=== Original 'Logged In' Descriptive Statistics ===")
print(f"Mean: {logged_mean:.3f}")
print(f"Median: {logged_median:.3f}")
print(f"Mode: {logged_mode}")
print(f"Standard Deviation: {logged_std:.3f}")
print(f"Range: {logged_range:.3f} (Min: {logged_min}, Max: {logged_max})")

print("\nValue counts:")
print(df['logged in'].value_counts(dropna=False))
print("\n Skewness:")
print(df['logged in'].skew())

print("\nFull describe():")
print(df['logged in'].describe())
print(f"Missing values: {df['logged in'].isnull().sum()}")

=== Original 'Logged In' Descriptive Statistics ===
Mean: 14.067
Median: 10.931
Mode: 0.0
Standard Deviation: 13.578
Range: 85.489 (Min: 0.0, Max: 85.48873672185293)

Value counts:
logged in
0.000000     106
NaN            5
10.380808      1
17.989603      1
13.770492      1
            ... 
2.874299       1
27.402300      1
16.071705      1
3.203513       1
25.152260      1
Name: count, Length: 391, dtype: int64

 Skewness:
1.1682113309

Full describe():
count    495.000000
mean      14.066915
std       13.577602
min        0.000000
25%        2.093709
50%       10.931016
75%       22.296099
max       85.488737
Name: logged in, dtype: float64
Missing values: 5


In [ ]:
plt.figure(figsize=(8, 2))

sns.boxplot(data=df, x='logged in', orient='h', color='lightblue', linewidth=1.0, linecolor='darkblue')
plt.title(f"Horizontal Boxplot: Logged In")
plt.xlabel('Logged In Value')
plt.tight_layout()
plt.show()

In [ ]:
# Mode imputation
if df['logged in'].isnull().sum() > 0:
    logged_mode_val = df['logged in'].mode()[0]
    df['logged in'] = df['logged in'].fillna(logged_mode_val)
    print(f"Imputed {df['logged in'].isnull().sum()} NaNs with mode: {logged_mode_val}")
else:
    print("No missing values to impute.")


# IQR on imputed data
Q1 = df['logged in'].quantile(0.25)
Q3 = df['logged in'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")

# Detect outliers
outliers_logged = df[(df['logged in'] < lower_bound) | (df['logged in'] > upper_bound)]
print("IQR outliers:")
print(outliers_logged[['logged in']])

# Cap outliers
df['logged_in_capped_iqr'] = df['logged in'].clip(lower=lower_bound, upper=upper_bound)

Imputed 0 NaNs with mode: 0.0
IQR bounds: [-28.7, 52.5]
IQR outliers:
     logged in
39   59.874187
52   57.714056
227  58.777472
245  54.964288
282  57.495773
379  57.620317
427  54.402603
453  85.488737
476  56.547914


In [ ]:
capped_mean = df['logged_in_capped_iqr'].mean()
capped_median = df['logged_in_capped_iqr'].median()
capped_std = df['logged_in_capped_iqr'].std()

print("\n=== 📊 Capped 'Logged In' Summary ===")
print(f"Mean: {capped_mean:.3f} (was {logged_mean:.3f})")
print(f"Median: {capped_median:.3f} (was {logged_median:.3f})")
print(f"Std: {capped_std:.3f} (was {logged_std:.3f})")
print("\nValue counts (capped):")
print(df['logged_in_capped_iqr'].value_counts())


=== 📊 Capped 'Logged In' Summary ===
Mean: 13.786 (was 14.067)
Median: 10.820 (was 10.931)
Std: 13.079 (was 13.578)

Value counts (capped):
logged_in_capped_iqr
0.000000     111
52.547759      9
5.299107       1
17.989603      1
13.770492      1
            ... 
7.064102       1
2.874299       1
27.402300      1
16.071705      1
25.152260      1
Name: count, Length: 382, dtype: int64


📊 **Observations for Logged In:**

* **Mean dropped 14.07 → 13.79 (0.2 pts):** Extreme hours no longer inflate average login time.

* **Median stable 10.93 → 10.82 (0.1 pts):** Core usage unchanged (capping targets tails only).

* **Std reduced 13.58 → 13.08 (3.7%):**
Variability tamed—outliers clipped to ~25-52 max (IQR bounds).

* **Still right-skewed but milder:** 111 zeros dominant, tail now capped vs exploding to 55+.

* **Value counts:** 111 non-users (0 hrs), diverse low-medium usage (2-27 hrs), 9 heavy users capped at 52.5 hrs.

### 🧩 Lessons

In [ ]:
# Key statistics for original 'Lessons'
lessons_mean = df['lessons'].mean()
lessons_median = df['lessons'].median()
lessons_mode = df['lessons'].mode()[0] if not df['lessons'].mode().empty else None
lessons_std = df['lessons'].std()
lessons_min, lessons_max = df['lessons'].min(), df['lessons'].max()
lessons_range = lessons_max - lessons_min

print("=== Original 'Lessons' Descriptive Statistics ===")
print(f"Mean: {lessons_mean:.3f}")
print(f"Median: {lessons_median:.3f}")
print(f"Mode: {lessons_mode}")
print(f"Standard Deviation: {lessons_std:.3f}")
print(f"Range: {lessons_range:.3f} (Min: {lessons_min}, Max: {lessons_max})")
print("\nFull describe:")
print(df['lessons'].describe())
print(f"Missing values: {df['lessons'].isnull().sum()}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Histogram with density curve
df['lessons'].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
ax.axvline(lessons_mean, color='red', linestyle='--', label=f'Mean: {lessons_mean:.1f}')
ax.axvline(lessons_median, color='green', linestyle='--', label=f'Median: {lessons_median:.1f}')
ax.axvline(lessons_mode, color='orange', linestyle='--', label=f'Mode: {lessons_mode}')
ax.legend()
ax.set_title("Lessons Histogram")
ax.set_xlabel('Number of Lessons')
ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

=== Original 'Lessons' Descriptive Statistics ===
Mean: 3.389
Median: 2.000
Mode: 2.0
Standard Deviation: 3.979
Range: 44.000 (Min: 0.0, Max: 44.0)

Full describe:
count    493.000000
mean       3.389452
std        3.978914
min        0.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       44.000000
Name: lessons, dtype: float64
Missing values: 7


In [ ]:
# Impute missing values
if df['lessons'].isnull().sum() > 0:
    lessons_mode_val = df['lessons'].mode()[0]
    df['lessons'] = df['lessons'].fillna(lessons_mode_val)
    print(f"Imputed {df['lessons'].isnull().sum()} NaNs with mode: {lessons_mode_val}")
else:
    print("No missing values to impute.")

Imputed 0 NaNs with mode: 2.0


In [ ]:
skew_lessons = df['lessons'].skew()

if skew_lessons > 0.5:
  print(f"Lessons skewness: {skew_lessons}")

Q1 = df['lessons'].quantile(0.25)
Q3 = df['lessons'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['lessons'] < lower) | (df['lessons'] > upper)][['lessons']]
print("IQR outliers:")
print(outliers.head())

df['lessons_capped_iqr'] = df['lessons'].clip(lower=lower, upper=upper)
method = 'IQR'

Lessons skewness: 3.603725994084447
IQR outliers:
    lessons
23     15.0
35     23.0
39     14.0
52     44.0
58     16.0


In [ ]:
capped_mean = df['lessons_capped_iqr'].mean()
capped_median = df['lessons_capped_iqr'].median()
capped_std = df['lessons_capped_iqr'].std()

print("\n=== Capped 'Lessons' Summary ===")
print(f"Mean: {capped_mean:.3f} (was {lessons_mean:.3f})")
print(f"Median: {capped_median:.3f} (was {lessons_median:.3f})")
print(f"Std: {capped_std:.3f} (was {lessons_std:.3f})")
print(f"Skew reduced: {df['lessons_capped_iqr'].skew():.2f} (was {skew_lessons:.2f})")
print("\nTop value counts (capped):")
print(df['lessons_capped_iqr'].value_counts().head(10))


=== Capped 'Lessons' Summary ===
Mean: 3.140 (was 3.389)
Median: 2.000 (was 2.000)
Std: 2.954 (was 3.979)
Skew reduced: 1.11 (was 3.60)

Top value counts (capped):
lessons_capped_iqr
2.0     99
1.0     90
0.0     89
3.0     45
5.0     44
4.0     39
6.0     25
11.0    24
8.0     21
7.0     15
Name: count, dtype: int64


In [ ]:
# Narrative observations (customize with your actual numbers)
capped_col = 'lessons_capped_iqr'

print("\n=== 📊 OBSERVATIONS for Lessons (Post-Capping) ===")
print(f"• Mean: {df[capped_col].mean():.1f} lessons (↓ from {lessons_mean:.1f})")
print(f"• Median: {df[capped_col].median():.1f} lessons (stable)")
print(f"• Top activity: {df[capped_col].mode()[0]:.0f} lessons (most common)")
print(f"• Variability: SD {df[capped_col].std():.1f} (↓{lessons_std - df[capped_col].std():.1f})")

skew_new = df[capped_col].skew()
print(f"• Shape: {'Mildly ' if abs(skew_new)<1 else ''}{'right' if skew_new>0 else 'left'}-skewed (skew={skew_new:.2f})")

# Business insights
q90 = df[capped_col].quantile(0.9)
print(f"• 90th percentile: {q90:.1f} → Most students <{q90:.0f} lessons")
print(f"• Engagement tiers: {len(df[df[capped_col]==0])} inactive, {len(df[df[capped_col]>df[capped_col].quantile(0.75)])} active")

print("\n✅ Capping success: Extremes tamed, core patterns preserved for modeling.")



=== 📊 OBSERVATIONS for Lessons (Post-Capping) ===
• Mean: 3.1 lessons (↓ from 3.4)
• Median: 2.0 lessons (stable)
• Top activity: 2 lessons (most common)
• Variability: SD 3.0 (↓1.0)
• Shape: right-skewed (skew=1.11)
• 90th percentile: 8.0 → Most students <8 lessons
• Engagement tiers: 89 inactive, 94 active

✅ Capping success: Extremes tamed, core patterns preserved for modeling.


### 🧩 Posts

In [ ]:
# Key statistics for original 'Posts'
posts_mean = df['posts'].mean()
posts_median = df['posts'].median()
posts_mode = df['posts'].mode()[0] if not df['posts'].mode().empty else None
posts_std = df['posts'].std()
posts_min, posts_max = df['posts'].min(), df['posts'].max()
posts_range = posts_max - posts_min

print("=== Original 'Posts' Descriptive Statistics ===")
print(f"Mean: {posts_mean:.3f}")
print(f"Median: {posts_median:.3f}")
print(f"Mode: {posts_mode}")
print(f"Standard Deviation: {posts_std:.3f}")
print(f"Range: {posts_range:.3f} (Min: {posts_min}, Max: {posts_max})")
print("\nValue counts (top 10):")
print(df['posts'].value_counts().head(10))
print("\nFull describe:")
print(df['posts'].describe())
print(f"Missing values: {df['posts'].isnull().sum()}")

=== Original 'Posts' Descriptive Statistics ===
Mean: 5.358
Median: 5.000
Mode: 5.0
Standard Deviation: 2.399
Range: 14.000 (Min: 0.0, Max: 14.0)

Value counts (top 10):
posts
5.0     87
3.0     67
6.0     67
4.0     67
7.0     65
8.0     44
2.0     30
1.0     19
9.0     18
10.0    14
Name: count, dtype: int64

Full describe:
count    495.000000
mean       5.357576
std        2.399270
min        0.000000
25%        4.000000
50%        5.000000
75%        7.000000
max       14.000000
Name: posts, dtype: float64
Missing values: 5


In [ ]:
plt.figure(figsize=(6, 2))

# Boxplot
sns.boxplot(data=df, x='posts', orient='h', color='lightcoral')
plt.title("Posts: Horizontal Boxplot")
plt.xlabel('Number of Posts')
plt.show()

In [ ]:
# Impute posts missing values
if df['posts'].isnull().sum() > 0:
    posts_mode_val = df['posts'].mode()[0]
    df['posts'] = df['posts'].fillna(posts_mode_val)
    print(f"Imputed {df['posts'].isnull().sum()} NaNs with mode: {posts_mode_val}")
else:
    print("No missing values to impute.")

Imputed 0 NaNs with mode: 5.0


In [ ]:
# 1. Calculate IQR bounds
Q1 = df['posts'].quantile(0.25)
Q3 = df['posts'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 2. IQR Capping (Winsorization)
# Note: Since 'posts' usually can't be negative, the lower bound often hits 0
df['posts_capped_iqr'] = df['posts'].clip(lower=lower_bound, upper=upper_bound)

# 3. PRINT RESULTS
print("\n=== IQR Capping Results ===")
print(f"Q1: {Q1:.1f}, Q3: {Q3:.1f}, IQR: {IQR:.1f}")
print(f"IQR-bounds applied: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Outliers clipped: {len(df[(df['posts'] < lower_bound) | (df['posts'] > upper_bound)])} rows")

print(f"\nOriginal: mean={posts_mean:.3f}, std={posts_std:.3f}, max={df['posts'].max():.1f}")
print(f"Capped:   mean={df['posts_capped_iqr'].mean():.3f}, std={df['posts_capped_iqr'].std():.3f}, max={df['posts_capped_iqr'].max():.1f}")

print("\nBefore/After extremes:")
print("Original max rows:")
print(df.nlargest(3, 'posts')[['posts']])
print("\nCapped max rows:")
print(df.nlargest(3, 'posts_capped_iqr')[['posts_capped_iqr']])


=== IQR Capping Results ===
Q1: 4.0, Q3: 7.0, IQR: 3.0
IQR-bounds applied: [-0.5, 11.5]
Outliers clipped: 2 rows

Original: mean=5.358, std=2.399, max=14.0
Capped:   mean=5.348, std=2.369, max=11.5

Before/After extremes:
Original max rows:
     posts
112   14.0
13    12.0
23    11.0

Capped max rows:
     posts_capped_iqr
13               11.5
112              11.5
23               11.0


In [1]:
capped_col = 'posts_capped_iqr'
skew_posts = df['posts'].skew()

print(f"\n=== 📊 POSTS OBSERVATIONS (Post-IQR Capping) ===")
print(f"• Mean: {df[capped_col].mean():.1f} posts (was {posts_mean:.1f})")
print(f"• Median: {df[capped_col].median():.1f} posts")
print(f"• Mode: {df[capped_col].mode()[0]:.0f} posts")
print(f"• Std: {df[capped_col].std():.1f} (was {posts_std:.1f})")
print(f"• Skew: {df[capped_col].skew():.2f} (was {skew_posts:.2f})")
print(f"• 90th percentile: {df[capped_col].quantile(0.9):.0f}")

print("\nTop engagement tiers (Capping frequency):")
print(df[capped_col].value_counts().head())

print(f"\n• Inactive (0 posts): {len(df[df[capped_col]==0])} ({len(df[df[capped_col]==0])/len(df)*100:.0f}%)")
print(f"• Active (≥ median): {len(df[df[capped_col]>=df[capped_col].median()])}")

print("\n✅ Ready for modeling: Non-parametric extremes capped.")

NameError: name 'df' is not defined

### 🧩 Score

In [ ]:
# Key statistics for original 'Score'
score_mean = df['score'].mean()
score_median = df['score'].median()
score_mode = df['score'].mode()[0] if not df['score'].mode().empty else None
score_std = df['score'].std()
score_min, score_max = df['score'].min(), df['score'].max()
score_range = score_max - score_min

print("=== Original 'Score' Descriptive Statistics ===")
print(f"Mean: {score_mean:.3f}")
print(f"Median: {score_median:.3f}")
print(f"Mode: {score_mode}")
print(f"Standard Deviation: {score_std:.3f}")
print(f"Range: {score_range:.3f} (Min: {score_min}, Max: {score_max})")
print("\nValue counts (top 10):")
print(df['score'].value_counts().head(10))
print("\nFull describe:")
print(df['score'].describe())
print(f"Missing values: {df['score'].isnull().sum()}")

=== Original 'Score' Descriptive Statistics ===
Mean: 34.999
Median: 32.137
Mode: 0.687008299232193
Standard Deviation: 15.630
Range: 82.152 (Min: 0.687008299232193, Max: 82.83908894616035)

Value counts (top 10):
score
38.749771    1
44.452539    1
18.985095    1
52.182803    1
24.172925    1
39.749603    1
18.550018    1
25.175535    1
19.454816    1
44.009339    1
Name: count, dtype: int64

Full describe:
count    495.000000
mean      34.998790
std       15.630228
min        0.687008
25%       22.772791
50%       32.137461
75%       46.992596
max       82.839089
Name: score, dtype: float64
Missing values: 5


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14))

# 1. Raw Posts vs Raw Score (ALL points, NaNs invisible)
sns.scatterplot(data=df, x='posts', y='score', ax=ax1, alpha=0.6, s=60)
sns.regplot(data=df, x='posts', y='score', ax=ax1, scatter=False,
            line_kws={'color':'red', 'lw':3}, robust=True)  # robust=True ignores outliers
ax1.set_title('Raw Posts vs Raw Score', fontweight='bold')
ax1.set_xlabel('posts')
ax1.axhline(y=60, color='orange', linestyle='--', alpha=0.7)

# 2. Raw Lessons vs Raw Score
sns.scatterplot(data=df, x='lessons', y='score', ax=ax2, alpha=0.6, s=60)
sns.regplot(data=df, x='lessons', y='score', ax=ax2, scatter=False,
            line_kws={'color':'red', 'lw':3}, robust=True)
ax2.set_title('Raw Lessons vs Raw Score', fontweight='bold')
ax2.set_xlabel('lessons')
ax2.axhline(y=60, color='orange', linestyle='--', alpha=0.7)

plt.suptitle('Full Engagement Analysis (Raw Data)', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
if df['score'].isnull().sum() > 0:
    score_mode_val = df['score'].mode()[0]
    df['score'] = df['score'].fillna(score_mode_val)
    print(f"Imputed {df['score'].isnull().sum()} NaNs with mode: {score_mode_val}")
else:
    print("No missing values to impute.")

Imputed 0 NaNs with mode: 0.687008299232193


In [ ]:
score_mean = df['score'].mean()
score_std = df['score'].std()
skew_score = df['score'].skew()

print(f"Score skewness: {skew_score:.3f}")

# Using Z-score
zscores = stats.zscore(df['score'])
df['score_capped_z'] = np.clip(zscores, -3, 3) * score_std + score_mean

print("\n=== Z-Score Capping Results ===")
print(f"Original: mean={score_mean:.3f}, std={score_std:.3f}, max={df['score'].max():.1f}")
print(f"Capped:   mean={df['score_capped_z'].mean():.3f}, std={df['score_capped_z'].std():.3f}, max={df['score_capped_z'].max():.1f}")
print(f"Outliers clipped: {len(df[np.abs(zscores)>3])} rows")
print(f"Z-bounds: [{score_mean-3*score_std:.1f}, {score_mean+3*score_std:.1f}]")


Score skewness: 0.381

=== Z-Score Capping Results ===
Original: mean=34.656, std=15.923, max=82.8
Capped:   mean=34.655, std=15.936, max=82.4
Outliers clipped: 1 rows
Z-bounds: [-13.1, 82.4]


In [ ]:
print("\n=== 📊 SCORE OBSERVATIONS (Post-Capping) ===")
print(f"• Mean: {df['score_capped_z'].mean():.1f}% (was {score_mean:.1f}%)")
print(f"• Median: {df['score_capped_z'].median():.1f}%")
print(f"• Mode: {df['score_capped_z'].mode()[0]:.0f}%")
print(f"• Std: {df['score_capped_z'].std():.1f} (was {score_std:.1f})")
print(f"• Skew: {df['score_capped_z'].skew():.2f} (was {skew_score:.2f})")
print(f"• 90th percentile: {df['score_capped_z'].quantile(0.9):.0f}%")

print("\nTop score tiers:")
print(df['score_capped_z'].value_counts().head())
print(f"\n• Failing (<60%): {len(df[df['score_capped_z']<60])} ({len(df[df['score_capped_z']<60])/len(df)*100:.0f}%)")
print(f"• Passing (≥60%): {len(df[df['score_capped_z']>=60])}")
print(f"• Honors (≥80%): {len(df[df['score_capped_z']>=80])}")

print("\n✅ Model-ready: Extremes capped, performance tiers clear.")


=== 📊 SCORE OBSERVATIONS (Post-Capping) ===
• Mean: 34.7% (was 34.7%)
• Median: 31.9%
• Mode: 1%
• Std: 15.9 (was 15.9)
• Skew: 0.38 (was 0.38)
• 90th percentile: 57%

Top score tiers:
score_capped_z
0.652989     6
54.177451    1
49.153849    1
45.149311    1
26.979190    1
Name: count, dtype: int64

• Failing (<60%): 461 (92%)
• Passing (≥60%): 39
• Honors (≥80%): 2

✅ Model-ready: Extremes capped, performance tiers clear.


### 🧩 Assignments

In [ ]:
# Key statistics for original 'Assignments'
assign_mean = df['assignments'].mean()
assign_median = df['assignments'].median()
assign_mode = df['assignments'].mode()[0] if not df['assignments'].mode().empty else None
assign_std = df['assignments'].std()
assign_min, assign_max = df['assignments'].min(), df['assignments'].max()
assign_range = assign_max - assign_min

print("=== Original 'Assignments' Descriptive Statistics ===")
print(f"Mean: {assign_mean:.3f}")
print(f"Median: {assign_median:.3f}")
print(f"Mode: {assign_mode}")
print(f"Standard Deviation: {assign_std:.3f}")
print(f"Range: {assign_range:.3f} (Min: {assign_min}, Max: {assign_max})")
print("\nValue counts (top 10):")
print(df['assignments'].value_counts().head(10))
print("\nFull describe():")
print(df['assignments'].describe())
print(f"Missing values: {df['assignments'].isnull().sum()}")

=== Original 'Assignments' Descriptive Statistics ===
Mean: 0.561
Median: 0.000
Mode: 0.0
Standard Deviation: 1.312
Range: 9.000 (Min: 0.0, Max: 9.0)

Value counts (top 10):
assignments
0.0    360
1.0     67
2.0     27
3.0     15
4.0      8
5.0      4
7.0      3
9.0      3
8.0      1
Name: count, dtype: int64

Full describe():
count    488.000000
mean       0.561475
std        1.311891
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        9.000000
Name: assignments, dtype: float64
Missing values: 12


In [ ]:
# Horizontal boxplot
plt.figure(figsize=(6, 2))

sns.boxplot(data=df, x='assignments', orient='h', color='violet')
plt.title("Assignments: Horizontal Boxplot")
plt.xlabel('Number of Assignments')

plt.tight_layout()
plt.show()

# Impute missing values
if df['assignments'].isnull().sum() > 0:
    assign_mode_val = df['assignments'].mode()[0]
    df['assignments'] = df['assignments'].fillna(assign_mode_val)
    print(f"Imputed {df['assignments'].isnull().sum()} NaNs with mode: {assign_mode_val}")
else:
    print("No missing values to impute.")

In [ ]:
# Manual Z-score selection
assign_mean = df['assignments'].mean()
assign_std = df['assignments'].std()
skew_assign = df['assignments'].skew()

print(f"Assignments skewness: {skew_assign:.3f}")

# IQR for high skew (3.64)
Q1 = df['assignments'].quantile(0.25)
Q3 = df['assignments'].quantile(0.75)
IQR = Q3 - Q1
lower = max(0, Q1 - 1.5 * IQR)  # Assignments ≥0
upper = Q3 + 1.5 * IQR

outliers = df[(df['assignments'] > upper)]  # Focus right tail
print(f"IQR bounds: [{lower:.1f}, {upper:.1f}]")
print(f"High outliers (> {upper:.1f}): {len(outliers)}")

df['assignments_capped_iqr'] = df['assignments'].clip(lower=lower, upper=upper)

print("\n=== IQR Capping Results ===")
print(f"Original max: {df['assignments'].max():.1f}")
print(f"Capped max: {df['assignments_capped_iqr'].max():.1f}")
print(df.nlargest(3, 'assignments')[['assignments']])
print(df.nlargest(3, 'assignments_capped_iqr')[['assignments_capped_iqr']])

Assignments skewness: 3.638
IQR bounds: [0.0, 2.5]
High outliers (> 2.5): 34

=== IQR Capping Results ===
Original max: 9.0
Capped max: 2.5
     assignments
151          9.0
397          9.0
441          9.0
    assignments_capped_iqr
58                     2.5
70                     2.5
75                     2.5


In [ ]:
print("\n=== 📊 ASSIGNMENTS OBSERVATIONS (IQR Capping) ===")
print(f"• Mean: {df['assignments_capped_iqr'].mean():.1f} (was {assign_mean:.1f})")
print(f"• Median: {df['assignments_capped_iqr'].median():.1f} (stable)")
print(f"• Mode: {df['assignments_capped_iqr'].mode()[0]:.0f}")
print(f"• Std: {df['assignments_capped_iqr'].std():.1f} (was {assign_std:.1f})")
print(f"• Skew: {df['assignments_capped_iqr'].skew():.2f} (was 3.64)")
print(f"• 90th percentile: {df['assignments_capped_iqr'].quantile(0.9):.0f}")
print("\nTop completion tiers:")
print(df['assignments_capped_iqr'].value_counts().head())
print(f"\n• Low (<median): {len(df[df['assignments_capped_iqr']<df['assignments_capped_iqr'].median()])}")
print(f"• High (≥median): {len(df[df['assignments_capped_iqr']>=df['assignments_capped_iqr'].median()])}")
print(f"• Max capped at: {df['assignments_capped_iqr'].max():.0f} (IQR upper bound)")

print("\n✅ IQR success: Extreme tail (20+) clipped, core patterns preserved.")


=== 📊 ASSIGNMENTS OBSERVATIONS (IQR Capping) ===
• Mean: 0.4 (was 0.5)
• Median: 0.0 (stable)
• Mode: 0
• Std: 0.8 (was 1.3)
• Skew: 1.73 (was 3.64)
• 90th percentile: 2

Top completion tiers:
assignments_capped_iqr
0.0    372
1.0     67
2.5     34
2.0     27
Name: count, dtype: int64

• Low (<median): 0
• High (≥median): 500
• Max capped at: 2 (IQR upper bound)

✅ IQR success: Extreme tail (20+) clipped, core patterns preserved.


### 🧩 Orientation

In [ ]:
# Key statistics for binary 'Orientation' (0/1)
orient_mean = df['orientation'].mean()
orient_median = df['orientation'].median()
orient_mode = df['orientation'].mode()[0] if not df['orientation'].mode().empty else None
orient_std = df['orientation'].std()

print("=== Original 'Orientation' Descriptive Statistics ===")
print(f"Mean (Orientation rate): {orient_mean:.3f} ({orient_mean*100:.1f}%)")
print(f"Median: {orient_median}")
print(f"Mode: {orient_mode}")
print(f"Standard Deviation: {orient_std:.3f}")
print("\nValue counts:")
print(df['orientation'].value_counts(dropna=False))
print("\nFull describe():")
print(df['orientation'].describe())
print(f"Missing values: {df['orientation'].isnull().sum()}")

=== Original 'Orientation' Descriptive Statistics ===
Mean (Orientation rate): 0.619 (61.9%)
Median: 1.0
Mode: 1.0
Standard Deviation: 0.486

Value counts:
orientation
1.0    306
0.0    188
NaN      6
Name: count, dtype: int64

Full describe():
count    494.000000
mean       0.619433
std        0.486018
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: orientation, dtype: float64
Missing values: 6


In [ ]:
plt.figure(figsize=(8, 6))
df['orientation'].value_counts().plot(kind='bar', color=['lightgreen', 'lightcoral'])
plt.title("Orientation: Counts")
plt.xlabel('Orientation')
plt.ylabel('Count')
for i, v in enumerate(plt.gca().containers[0]):
    plt.gca().text(i, v.get_height() + 1, str(int(v.get_height())), ha='center')

plt.tight_layout()
plt.show()

print("\nCount for orientation = 1:", df[df['orientation'] == 1].value_counts().sum())
print("Count for orientation = 0:", df[df['orientation'] == 0].value_counts().sum())

# Impute missing values
if df['orientation'].isnull().sum() > 0:
    orient_mode_val = df['orientation'].mode()[0]
    df['orientation'] = df['orientation'].fillna(orient_mode_val)
    print(f"Imputed {df['orientation'].isnull().sum()} NaNs with mode: {orient_mode_val}")
else:
    print("No missing values to impute.")


Count for orientation = 1: 303
Count for orientation = 0: 185
Imputed 0 NaNs with mode: 1.0


In [ ]:
print("\n=== 📊 ORIENTATION OBSERVATIONS ===")
print(f"• Orientation rate: {df['orientation'].mean():.1%} ({df['orientation'].sum()}/{len(df)} students)")
print(f"• Non-oriented: {(1-df['orientation'].mean()):.1%}")
print(f"• Mode: {df['orientation'].mode()[0]} ({df['orientation'].value_counts().max()}/{len(df):.0f} = {df['orientation'].value_counts(normalize=True).max():.1%})")
print("\nDistribution:")
print(df['orientation'].value_counts(normalize=True).round(3))


=== 📊 ORIENTATION OBSERVATIONS ===
• Orientation rate: 62.4% (312.0/500 students)
• Non-oriented: 37.6%
• Mode: 1.0 (312/500 = 62.4%)

Distribution:
orientation
1.0    0.624
0.0    0.376
Name: proportion, dtype: float64


In [ ]:
print("="*70)
print("We have dealt with all our missing values.")
print("="*70)
df.info()

We have dealt with all our missing values.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              500 non-null    int64  
 1   sex                     500 non-null    object 
 2   age                     494 non-null    float64
 3   country                 500 non-null    object 
 4   logged in               500 non-null    float64
 5   lessons                 500 non-null    float64
 6   assignments             500 non-null    float64
 7   posts                   500 non-null    float64
 8   mentoring               500 non-null    object 
 9   score                   500 non-null    float64
 10  orientation             500 non-null    float64
 11  age_imputed             500 non-null    float64
 12  age_capped_iqr          500 non-null    float64
 13  logged_in_capped_iqr    500 non-null    float64
 14 

### 🛠️ **Step 3: Data Pre-Processing**

In this section, we're figuring out which student characteristics actually matter for predicting scores. We test each feature (like `age`, `lessons completed`, or `country`) to see if it has a real, statistically meaningful relationship with student performance — or if any pattern we see could just be random chance. Features that pass this test move forward; those that don't are set aside to keep our model focused and accurate.

In [ ]:
from scipy.stats import pearsonr, pointbiserialr, f_oneway

### 🧪 **NULL HYPOTHESIS**

There is no statistically significant relationship (or difference in means) between each feature — age, lessons, logins, posts, assignments, orientation, country, sex, and mentoring — and student score `score_capped_z`.

**Per test type:**

* Pearson / Point-Biserial: There is no linear correlation between [feature] and score `(r = 0)`.
* ANOVA: The mean score is equal across all groups of [country / sex / mentoring].

Features where `p < 0.05` **lead to rejecting H₀**, indicating a statistically significant relationship worth retaining for modeling.



In [ ]:
# Ensure numeric columns
def safe_numeric(series):
    return pd.to_numeric(series, errors='coerce').dropna()

target_cont = 'score_capped_z'
results = []

# 1. Numeric features (Pearson)
num_feats = ['age_capped_iqr', 'lessons_capped_iqr', 'logged_in_capped_iqr',
             'posts_capped_iqr', 'assignments_capped_iqr']

for feat in num_feats:
    x = safe_numeric(df[feat])
    y = safe_numeric(df[target_cont])
    if len(x) > 3 and len(y) > 3:  # Min sample
        r, p = pearsonr(x, y)
        results.append({'Feature': feat, 'Test': 'Pearson', 'P Result': p, 'Reject_H0': p<0.05})

# 2. Binary categorical
bin_cats = ['orientation']
for feat in bin_cats:
    x = safe_numeric(df[feat])  # Converts 0/1
    y = safe_numeric(df[target_cont])
    if len(x) > 3 and len(y) > 3 and x.nunique() == 2:  # Binary check
        r_pb, p_pb = pointbiserialr(x, y)
        results.append({'Feature': feat, 'Test': 'Point-Biserial', 'P Result': p_pb, 'Reject_H0': p_pb<0.05})

# 3. Country (ANOVA)
if 'country' in df.columns:
    groups = [safe_numeric(df[df['country']==c][target_cont]) for c in df['country'].unique() if len(df[df['country']==c]) > 1]
    if len(groups) > 1 and all(len(g) > 1 for g in groups):
        f_stat, p_anova = f_oneway(*groups)
        results.append({'Feature': 'country', 'Test': 'ANOVA', 'P Result': p_anova, 'Reject_H0': p_anova<0.05})


# 4. Sex vs continuous score (ANOVA)
groups = [df[df['sex'] == cat]['score_capped_z'].dropna()
          for cat in df['sex'].unique() if pd.notna(cat)]
if all(len(g) > 1 for g in groups):
    f_stat, p_anova = f_oneway(*groups)
    results.append({'Feature': 'sex', 'Test': 'ANOVA', 'P Result': p_anova, 'Reject_H0': p_anova<0.05})

# 5. Mentoring (ANOVA)
if 'mentoring' in df.columns:
    groups = [safe_numeric(df[df['mentoring']==c][target_cont])
              for c in df['mentoring'].unique() if len(df[df['mentoring']==c]) > 1]
    if len(groups) > 1 and all(len(g) > 1 for g in groups):
        f_stat, p_anova = f_oneway(*groups)
        results.append({'Feature': 'mentoring', 'Test': 'ANOVA', 'P Result': p_anova, 'Reject_H0': p_anova<0.05})

# Results
results_df = pd.DataFrame(results).round(4)
print("✅ Feature Tests (p<0.05 = Significant):\n", results_df)
print("\n 📌 Keep these for modeling:\n", results_df[results_df['Reject_H0'] == True]['Feature'].tolist())


✅ Feature Tests (p<0.05 = Significant):
                   Feature            Test  P Result  Reject_H0
0          age_capped_iqr         Pearson    0.0536      False
1      lessons_capped_iqr         Pearson    0.0000       True
2    logged_in_capped_iqr         Pearson    0.0000       True
3        posts_capped_iqr         Pearson    0.0000       True
4  assignments_capped_iqr         Pearson    0.0000       True
5             orientation  Point-Biserial    0.0001       True
6                 country           ANOVA    0.3458      False
7                     sex           ANOVA    0.6984      False
8               mentoring           ANOVA    0.0000       True

 📌 Keep these for modeling:
 ['lessons_capped_iqr', 'logged_in_capped_iqr', 'posts_capped_iqr', 'assignments_capped_iqr', 'orientation', 'mentoring']


### 🏆 **Summary of Hypothesis Testing Results**

* **Rejected H₀ (p < 0.05):** For `lessons_capped_iqr`, `logged_in_capped_iqr`, `posts_capped_z`, `assignments_capped_iqr`, `orientation`, and `mentoring`, the p-value fell below 0.05, so we reject the null hypothesis and conclude that there is a statistically significant relationship between these features and student score. This means there is **ENOUGH evidence** to prove that these features have a meaningful association with student engagement.
---

* **Failed to Reject H₀ (p ≥ 0.05):** For `age_capped_iqr`, `country`, and `sex`, the p-value exceeded 0.05, so we fail to reject the null hypothesis and conclude that there is no statistically significant relationship between these features and student score. This means there is **NOT ENOUGH** evidence to claim that these features have a meaningful association with student engagement.

### 🛠️ **Step 3.1 Feature Selection**

Here we narrow down our list of features even further using two approaches: checking whether any features are so similar to each other that keeping both would be redundant, and using a Random Forest model to rank which features are actually most useful for making predictions. We ended up keeping **5 key features**: `logins`, `lessons`, `posts`, `assignments`, and `mentoring` status.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import chi2, VarianceThreshold, SelectFromModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# 1. CORRELATION HEATMAP + HIGH CORR DETECTION
numerical_features = ['age_capped_iqr', 'logged_in_capped_iqr', 'lessons_capped_iqr',
                      'posts_capped_iqr', 'score_capped_z', 'assignments_capped_iqr']

plt.figure(figsize=(10, 8))
corr_matrix = df[numerical_features].corr()

# Highlight high correlations
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})

plt.title('Correlation Matrix (High corr > 0.7 flagged)', fontweight='bold')
plt.tight_layout()
plt.show()

# 2. High correlation pairs (>0.7)
high_corr_pairs = []
to_drop_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))
            # Keep first, drop second
            if corr_matrix.columns[j] not in to_drop_corr:
                to_drop_corr.append(corr_matrix.columns[j])

print("🔥 High correlations (>0.7):")
for pair in high_corr_pairs:
    print(f"  {pair[0]} ↔ {pair[1]}: {pair[2]:.3f}")
print(f"Features to drop due to high correlation: {to_drop_corr}")

# 3. CHI-SQUARE: Categorical vs discretized score (ROBUST)
df['score_binned'] = pd.cut(df['score_capped_z'], bins=5, labels=False)

categorical_features = ['country', 'sex', 'mentoring', 'orientation']

X_cat = pd.get_dummies(df[categorical_features], drop_first=True)
y_binned = df['score_binned']

# Chi-square
chi_scores, chi_pvals = chi2(X_cat, y_binned)
chi_df = pd.DataFrame({
    'Feature': X_cat.columns.str.split('_').str[0],
    'Chi2_Score': chi_scores,
    'P_Value': chi_pvals
}).groupby('Feature').min()

print("\n📊 Chi-square results (p<0.05 = significant):")
print(chi_df.sort_values('P_Value'))

🔥 High correlations (>0.7):
Features to drop due to high correlation: []

📊 Chi-square results (p<0.05 = significant):
             Chi2_Score       P_Value
Feature                              
mentoring    179.297997  1.055110e-37
orientation   12.397792  1.462580e-02
sex            1.785197  1.413340e-01
country        1.762125  3.949133e-01


**Chi-Square Test Result Explanation:**

`mentoring` — **extremely significant** (p ≈ 1.05e-37). This is by far the strongest predictor. A chi2 score of 179 is massive, meaning the distribution of student score variable changes dramatically depending on the mentoring category. This feature will be kept for modeling.

`orientation` — **significant** (p ≈ 0.015). A real but much weaker relationship with the target. Might be included in the model, but it explains far less than mentoring.

`sex` — **not significant** (p ≈ 0.14). The relationship could easily be due to random chance. This feature will not be included.

`country` — **not significant** (p ≈ 0.39). Weakest result of the four. No detectable association with the target in this dataset. This feature will not be included.

In [ ]:
numerical_feats = ['age_capped_iqr', 'logged_in_capped_iqr',
                      'lessons_capped_iqr', 'posts_capped_iqr',
                      'assignments_capped_iqr']

categorical_feats = ['country', 'sex', 'mentoring', 'orientation']

# Target
y = df['score_binned']

# Combine all features
X_all = pd.get_dummies(df[numerical_feats + categorical_feats], drop_first=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

# After splitting (X_train is DataFrame)
rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=42)
rf.fit(X_train.values, y_train.values)  # Use .values to drop names

# SelectFromModel (prefit=True; uses fitted RF)
selector = SelectFromModel(rf, prefit=True)
X_train_transformed = selector.transform(X_train.values)

# Get selected features
original_features = X_all.columns
features_bool = selector.get_support()
selected_features = original_features[features_bool].tolist()
print(f"Original features: {len(original_features)}")
print(f"Selected features ({len(selected_features)}): {selected_features}")

# Plot importances
feature_importance = pd.DataFrame({
    "feature": selected_features,
    "importance": rf.feature_importances_[features_bool]
}).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance["feature"], feature_importance["importance"])
plt.xlabel("Importance")
plt.title("Selected Feature Importances from Random Forest")
plt.tight_layout()
plt.show()

Original features: 15
Selected features (5): ['logged_in_capped_iqr', 'lessons_capped_iqr', 'posts_capped_iqr', 'assignments_capped_iqr', 'mentoring_Yes']


In [ ]:
# Create final DataFrames with selected features only (drop excluded)
X_train_selected = selector.transform(X_train.values)
X_test_selected = selector.transform(X_test.values)

# Convert to DataFrames with feature names
X_train_final = pd.DataFrame(X_train_selected, columns=selected_features, index=X_train.index)
X_test_final = pd.DataFrame(X_test_selected, columns=selected_features, index=X_test.index)

print(f"\n✅ FINAL DATAFRAMES CREATED:")
print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(f"Features kept: {selected_features}")
print(f"Features dropped ({len(original_features) - len(selected_features)}): {[f for f in original_features if f not in selected_features]}")

print("\n📊 X_train_final:")
print(X_train_final.head())

# Verify no target leakage and ready for modeling
print(f"\ny_train shape: {y_train.shape}, dtype: {y_train.dtype}")
print("✅ Ready for encoding!")


✅ FINAL DATAFRAMES CREATED:
X_train_final shape: (400, 5)
X_test_final shape: (100, 5)
Features kept: ['logged_in_capped_iqr', 'lessons_capped_iqr', 'posts_capped_iqr', 'assignments_capped_iqr', 'mentoring_Yes']
Features dropped (10): ['age_capped_iqr', 'orientation', 'country_France', 'country_Germany', 'country_Italy', 'country_Other', 'country_Spain', 'sex_Male', 'sex_Non-Binary', 'sex_Prefer not to say']

📊 X_train_final:
    logged_in_capped_iqr lessons_capped_iqr posts_capped_iqr  \
249                  0.0                0.0              9.0   
433            10.646859                0.0              1.0   
19             14.594025                8.0              8.0   
322             2.759814                2.0              5.0   
332            31.748792                1.0              7.0   

    assignments_capped_iqr mentoring_Yes  
249                    0.0         False  
433                    0.0         False  
19                     2.0          True  
322         

### 🛠️ **Step 3.2 Encoding Categorical Values**

Computers can only work with numbers, not text. This step converts categorical (yes/no or label-based) data — specifically the mentoring column — into a numerical format (0s and 1s) so our machine learning models can understand and use it.

In [ ]:
# Align both with train's columns post-dummies
if 'mentoring_Yes' in X_train_final.columns:  # Check exact column
    X_train_final = pd.get_dummies(
        X_train_final, columns=['mentoring_Yes'], drop_first=True, dtype=int
    )
    # CRITICAL: Apply to test separately, then align columns
    X_test_final = pd.get_dummies(
        X_test_final, columns=['mentoring_Yes'], drop_first=True, dtype=int
    )
    # Match test to train (adds missing 0s, drops extras)
    X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)

print(X_train_final['mentoring_Yes_True'].head())  # Now exists in both
print("Train cols:", X_train_final.columns.tolist())
print("Test cols:", X_test_final.columns.tolist())

249    0
433    0
19     1
322    0
332    1
Name: mentoring_Yes_True, dtype: int64
Train cols: ['logged_in_capped_iqr', 'lessons_capped_iqr', 'posts_capped_iqr', 'assignments_capped_iqr', 'mentoring_Yes_True']
Test cols: ['logged_in_capped_iqr', 'lessons_capped_iqr', 'posts_capped_iqr', 'assignments_capped_iqr', 'mentoring_Yes_True']


### 🛠️ **Step 3.3 Feature Scaling**

Different features are measured on very different scales (e.g., logins might range 0–100, while assignments range 0–5). If we leave them as-is, the model may unfairly weight larger-numbered features. Scaling puts all numerical features on the same playing field so no single feature dominates simply because of its unit size.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Fix dtypes for numeric features
num_features = [
    'logged_in_capped_iqr',
    'lessons_capped_iqr',
    'posts_capped_iqr',
    'assignments_capped_iqr'
]

# Convert to numeric (both sets)
for col in num_features:
    X_train_final[col] = pd.to_numeric(X_train_final[col], errors='coerce')
    X_test_final[col] = pd.to_numeric(X_test_final[col], errors='coerce')

# Scale: Fit on TRAIN ONLY, transform BOTH
scaler = StandardScaler()
X_train_final[num_features] = scaler.fit_transform(X_train_final[num_features])
X_test_final[num_features] = scaler.transform(X_test_final[num_features])

print("\n✅ Scaling complete (separate train/test):")
print("X_train_final dtypes:\n", X_train_final.dtypes)
print("===Sample train scaled:===\n", X_train_final[num_features].head())
print("===Sample test scaled:===\n", X_test_final[num_features].head())
print("========")
print(f"Train and test set shapes: {X_train_final.shape} {X_test_final.shape}")


✅ Scaling complete (separate train/test):
X_train_final dtypes:
 logged_in_capped_iqr      float64
lessons_capped_iqr        float64
posts_capped_iqr          float64
assignments_capped_iqr    float64
mentoring_Yes_True          int64
dtype: object
===Sample train scaled:===
      logged_in_capped_iqr  lessons_capped_iqr  posts_capped_iqr  \
249             -1.069304           -1.070220          1.574245   
433             -0.236652           -1.070220         -1.836441   
19               0.072042            1.684972          1.147909   
322             -0.853469           -0.381422         -0.131098   
332              1.413654           -0.725821          0.721573   

     assignments_capped_iqr  
249               -0.519953  
433               -0.519953  
19                 2.063665  
322               -0.519953  
332               -0.519953  
===Sample test scaled:===
      logged_in_capped_iqr  lessons_capped_iqr  posts_capped_iqr  \
361             -1.069304           -1.070220

### 🛠️ **Step 3.4 Feature Engineering**

Rather than just using the raw features we have, we create two new combined features to give the model richer signals: a `Total Activity` score (summing logins, lessons, assignments, and posts) and a `Mentoring × Activity interaction` (capturing whether mentored students who are also highly active perform especially well). These engineered features can help the model pick up on patterns it might otherwise miss.

In [ ]:
# Simplified & Stable Feature Engineering
for X_eng in [X_train_final, X_test_final]:

    # 1. Total Activity (Volume only)
    # We exclude posts here to keep this a "Pure Volume" metric.
    # We keep the raw components in the model so RF can still use them individually.
    X_eng['total_activity'] = (
        X_eng['logged_in_capped_iqr'] +
        X_eng['lessons_capped_iqr'] +
        X_eng['assignments_capped_iqr'] +
        X_eng['posts_capped_iqr']
    )

    # 2. Mentoring Interaction (High Signal)
    # Interaction terms are great, but we use the boolean 'mentoring_Yes_True' directly.
    X_eng['ment_act_interact'] = X_eng['mentoring_Yes_True'] * X_eng['total_activity']

### 🔬📈 **Step 4. Supervised Machine Learning Models**

Now that our data is clean and prepared, we put it to work. We test multiple model types — some that predict a student's exact score **(regression)**, and some that predict whether a student is highly engaged or not **(classification)**. By comparing how well each model performs, we can identify the best approach for understanding and predicting student outcomes.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report, balanced_accuracy_score
)

In [ ]:
print("🎯 REGRESSION vs CLASSIFICATION COMPARISON")
print("Target: score_capped_z vs highly_engaged")
print("="*70)

# 1. Create classification target (threshold 0.8 for "Highly Engaged")
y_train_clf = (y_train >= np.median(y_train)).astype(int)
y_test_clf = (y_test >= np.median(y_test)).astype(int)

print(f"Classification balance - Train: {y_train_clf.mean():.1%} positive")

# 2. Regression Models (predict continuous score)
reg_models = {
    'Linear Reg': LinearRegression(),
    'Decision Tree Reg': DecisionTreeRegressor(random_state=42, max_depth=5),
    'KNN Reg': KNeighborsRegressor(n_neighbors=5)
}

# 3. Classification Models (predict binary engaged/not)
clf_models = {
    'Decision Tree Clf': DecisionTreeClassifier(random_state=42, max_depth=5),
    'KNN Clf': KNeighborsClassifier(n_neighbors=5)
}

# 4. Train & Evaluate
results = []

# Regression
print("📈 REGRESSION RESULTS (score_capped_z)")
for name, model in reg_models.items():
    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({'Type': 'Regression', 'Model': name, 'R²': r2, 'RMSE': rmse})
    print(f"{name:15} | R²: {r2:.3f} | RMSE: {rmse:.3f}")

print("\n🎯 CLASSIFICATION RESULTS (highly_engaged)")
for name, model in clf_models.items():
    model.fit(X_train_final, y_train_clf)
    y_pred = model.predict(X_test_final)

    acc = accuracy_score(y_test_clf, y_pred)                 # standard
    bal_acc = balanced_accuracy_score(y_test_clf, y_pred)    # balanced

    results.append({'Type': 'Classification', 'Model': name, 'Accuracy': acc})
    print(f"{name:15} | Accuracy: {acc:.3f} | Balanced Accuracy: {bal_acc:.3f}")

# 5. Summary Table
results_df = pd.DataFrame(results)
print("\n🏆 FINAL COMPARISON")
print(results_df.pivot_table(index='Type', columns='Model', values=['R²', 'Accuracy', 'RMSE']).round(3))


🎯 REGRESSION vs CLASSIFICATION COMPARISON
Target: score_capped_z vs highly_engaged
Classification balance - Train: 88.8% positive
📈 REGRESSION RESULTS (score_capped_z)
Linear Reg      | R²: 0.762 | RMSE: 0.513
Decision Tree Reg | R²: 0.771 | RMSE: 0.503
KNN Reg         | R²: 0.777 | RMSE: 0.496

🎯 CLASSIFICATION RESULTS (highly_engaged)
Decision Tree Clf | Accuracy: 0.860 | Balanced Accuracy: 0.590
KNN Clf         | Accuracy: 0.880 | Balanced Accuracy: 0.751

🏆 FINAL COMPARISON
                        Accuracy                      RMSE                     \
Model          Decision Tree Clf KNN Clf Decision Tree Reg KNN Reg Linear Reg   
Type                                                                            
Classification              0.86    0.88               NaN     NaN        NaN   
Regression                   NaN     NaN             0.503   0.496      0.513   

                              R²                     
Model          Decision Tree Reg KNN Reg Linear Reg  
Typ

#### **🤓☝️ Supervised Machine Learning Models Insights**
We tested both regression (predicting the exact score) and classification (predicting whether a student is "highly engaged" or not). **All three** **regression models** explained `~76–78%` of the variation in scores `(R² ≈ 0.76–0.78)`, which is solid. For **classification**, KNN edged out the Decision Tree with `88%` accuracy, though the high imbalance in the data (89% of students labeled "positive") means raw accuracy is misleading — **balanced accuracy is a better judge here**.


### 🔬📈 **Step 4.1 Model Tuning - Regression**

Think of model tuning like adjusting the settings on a camera to get the sharpest photo. Our initial regression model works, but by systematically testing different configurations (like how deep the decision trees can grow), we find the combination that predicts student scores most accurately without over-memorising the training data.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor  # RandomForestClassifier already imported in Step 3.1
from sklearn.model_selection import GridSearchCV, cross_val_score, RepeatedKFold, RepeatedStratifiedKFold

In [ ]:
print("=" * 60)

# REGRESSION PIPELINE (prevents selector leakage)
selector_base_reg = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42, n_jobs=-1)
selector_reg = SelectFromModel(selector_base_reg)
rf_reg = RandomForestRegressor(random_state=42, n_jobs=-1)

pipe_reg = Pipeline([
    ('selector', selector_reg),
    ('rf', rf_reg)
])

# Tighter params (smaller trees, higher regularization)
reg_params = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [3, 4],  # Stricter
    'rf__min_samples_split': [10, 20],
    'rf__min_samples_leaf': [5, 10],
    'rf__max_features': ['sqrt', 0.8]
}

grid_reg = GridSearchCV(pipe_reg, reg_params, cv=5, scoring='r2', n_jobs=-1)
grid_reg.fit(X_train_final, y_train)
best_reg = grid_reg.best_estimator_

print(f"Best Reg Params: {grid_reg.best_params_}")
print(f"CV R²: {grid_reg.best_score_:.3f}")

# TRAIN-TEST GAP CHECK
train_r2 = best_reg.score(X_train_final, y_train)
test_r2 = r2_score(y_test, best_reg.predict(X_test_final))
gap_r2 = train_r2 - test_r2
print(f"Train R²: {train_r2:.3f} | Test R²: {test_r2:.3f} | Gap: {gap_r2:.3f}")
if gap_r2 > 0.15:
    print("⚠️  HIGH GAP: Further regularize (reduce max_depth)")

Best Reg Params: {'rf__max_depth': 3, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 10, 'rf__min_samples_split': 10, 'rf__n_estimators': 200}
CV R²: 0.640
Train R²: 0.684 | Test R²: 0.728 | Gap: -0.044


#### **🤓☝️ Model Tuning Insights (Reg)**
After tuning, the regression model achieved a cross-validated `R² of 0.640`, meaning it explains about 64% of score variation on unseen data. Importantly, the gap between training and test performance is very small (–0.044), which tells us **the model is not overfitting** — it generalises well to new students rather than just memorising the training data.

### 🔬📈 **Step 4.2 Model Tuning - Classification**

Same idea as regression tuning, but applied to our classification model. Here we're fine-tuning the settings that determine whether the model correctly identifies engaged vs. non-engaged students — with special care to treat both groups fairly, since highly engaged students are much more common in our dataset.

In [ ]:
print("=" * 60)

# CLASSIFICATION PIPELINE (for highly_engaged binarized target)
selector_base_clf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42, class_weight='balanced', n_jobs=-1)
selector_clf = SelectFromModel(selector_base_clf)
rf_clf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)

pipe_clf = Pipeline([
    ('selector', selector_clf),
    ('rf', rf_clf)
])

clf_params = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [3, 4],
    'rf__min_samples_split': [10, 20],
    'rf__min_samples_leaf': [5, 10],
    'rf__max_features': ['sqrt', 0.8]
}

grid_clf = GridSearchCV(pipe_clf, clf_params, cv=5, scoring='balanced_accuracy', n_jobs=-1)
grid_clf.fit(X_train_final, y_train_clf)
best_clf = grid_clf.best_estimator_

print(f"Best Clf Params: {grid_clf.best_params_}")
print(f"CV BalAcc: {grid_clf.best_score_:.3f}")

train_balacc = balanced_accuracy_score(y_train_clf, best_clf.predict(X_train_final))
test_balacc = balanced_accuracy_score(y_test_clf, best_clf.predict(X_test_final))
gap_balacc = train_balacc - test_balacc
print(f"Train BalAcc: {train_balacc:.3f} | Test BalAcc: {test_balacc:.3f} | Gap: {gap_balacc:.3f}")

Best Clf Params: {'rf__max_depth': 3, 'rf__max_features': 0.8, 'rf__min_samples_leaf': 5, 'rf__min_samples_split': 10, 'rf__n_estimators': 100}
CV BalAcc: 0.792
Train BalAcc: 0.850 | Test BalAcc: 0.907 | Gap: -0.057


#### 🤓☝️ **Model Tuning Insights (Clf)**

The tuned classification model achieved a balanced accuracy of `0.792` in cross-validation, meaning it correctly identifies both engaged and non-engaged students about 79% of the time even accounting for the class imbalance. Again, the train-test gap is minimal (–0.057), **confirming the model generalises well** and isn't overfit.

### **🔬📈 Step 4.3 Final Model Evaluation**

In [464]:
# 10x5 CV for stability
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
reg_cv_scores = cross_val_score(best_reg, X_train_final, y_train, cv=rkf, scoring='r2', n_jobs=-1)
print(f"Reg Robust CV R²: {reg_cv_scores.mean():.3f} (+/- {reg_cv_scores.std() * 2:.3f})")

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
clf_cv_scores = cross_val_score(best_clf, X_train_final, y_train_clf, cv=rskf, scoring='balanced_accuracy', n_jobs=-1)
print(f"Clf Robust CV BalAcc: {clf_cv_scores.mean():.3f} (+/- {clf_cv_scores.std() * 2:.3f})")

Reg Robust CV R²: 0.636 (+/- 0.167)


KeyboardInterrupt: 

#### **🤓☝️ Final Model Evaluation Insights**

Using a more rigorous 10×5 repeated cross-validation (testing the model 50 times on different data splits), the **regression model** averaged `R² = 0.636 ± 0.167` and the **classifier** averaged `balanced accuracy = 0.768 ± 0.191`. The wider variation (±) reminds us our dataset is small (500 students), so results can fluctuate depending on which students end up in the test set. Overall, **both models are reasonably stable** and performing well given the data size.

In [ ]:
print(f"🔍 OVERFITTING CHECK \n{'='*60}")
depths = [1, 2, 3]

tasks = {
    "Regression": dict(
        model=RandomForestRegressor,
        y_tr=y_train, y_te=y_test,
        metric=r2_score, label="R²",
        ax=axes[0], extra={}
    ),
    "Classification": dict(
        model=RandomForestClassifier,
        y_tr=y_train_clf, y_te=y_test_clf,
        metric=accuracy_score, label="Accuracy",
        ax=axes[1], extra={"class_weight": "balanced"}
    )
}

for name, t in tasks.items():
    train_scores, test_scores = [], []

    for d in depths:
        model = t["model"](
            random_state=42, max_depth=d, **t["extra"]
        ).fit(X_train_final, t["y_tr"])

        tr = t["metric"](t["y_tr"], model.predict(X_train_final))
        te = t["metric"](t["y_te"], model.predict(X_test_final))
        gap = tr - te

        status = (
            "🚨 SEVERE OVERFIT" if gap > 0.15 else
            "⚠️ OVERFITTING" if gap > 0.10 else
            "✅ GOOD"
        )

        print(f"{name} max_depth={d}\n Train: {tr:.3f} | Test: {te:.3f} | Gap: {gap:.3f} | {status}")

        train_scores.append(tr)
        test_scores.append(te)

#### **🤓☝️ What The Result Shows**
**Regression:**

All three depth settings pass the overfitting check with small, negative gaps (meaning the test score is actually slightly higher than training, which is healthy). The best performance is at max_depth=3 `(Test R² = 0.748)`, and increasing complexity consistently improves both train and test scores — suggesting the model still has room to learn without memorising the data.

**Classification:**

Similarly clean results across all depths. The gap grows slightly as depth increases `(from –0.028 to +0.013)`, but remains well within the safe zone. At max_depth=3, the model correctly classifies 84% of test cases, with train and test scores tracking closely together.

### 🥇 **Step 4.4 Model Selection**

> **Tuned RandomForestClassifier is the best model.**

Its test balanced accuracy of 0.907 `(CV 0.792, gap -0.057)` reliably handles the severe class imbalance (88.8% positive), outperforming regression's max test R² 0.777.

---
**Why Classification Over Regression**

Raw accuracy (0.880) misleads due to imbalance—a dummy classifier guessing "positive" hits ~89% without learning. Balanced accuracy penalizes this, yet tuned RF Clf excels `(test 0.907 > robust CV 0.768)`.

Regression R² `~0.77` seems strong but compares continuous prediction (harder) to binary; no equivalent "balanced R²" adjustment. Clf directly answers "high-risk dropout?" for interventions.

In [ ]:
!pip install datascience -q
from datascience import *

In [ ]:
results_table = Table().with_columns(
    "Metric", make_array("Test Score", "CV Score", "Train-Test Gap", "Robust CV"),
    "Random Forest Clf", make_array("BalAcc: 0.907", "0.792", "-0.057", "0.768 ± 0.191"),
    "Random Forest Reg", make_array("R²: 0.728", "R²: 0.640", "-0.044", "0.636 ± 0.167"),
    "Why Clf Wins", make_array(
        "Handles imbalance",
        "Stable generalization",
        "Low overfitting",
        "Consistent despite small data"
    )
)

results_table.show()

### 📝 **FINAL NOTE**

Please see the PDF submitted alongside this notebook for Model Communication results.

In [ ]:
conn.close()